In [5]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import time
from sklearn.preprocessing import StandardScaler
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


In [ ]:
stap_per_dag = 24 * 12
# aantal dagen lengte van de imput    
aantal_dagen_imput = 2             
imput_len = aantal_dagen_imput * stap_per_dag
# aantal dagen van de voorspelling
aantal_dagen_forecast = 3
horizon= aantal_dagen_forecast * stap_per_dag
stride = 12

TARGET_COL= "POW"
FEATURE_COLS= []  # alle features ook de target

HIDDEN_SIZE   = 64
NUM_LAYERS    = 2
DROPOUT       = 0.2
BATCH_SIZE    = 32
EPOCHS        = 30
LR            = 1e-3
PATIENCE      = 5

TRAIN_FRAC = 0.70 # percentage als train data
VAL_FRAC = 0.15  # percentage als validatie data

## DATA LADEN 
### eisen: opgeschoond en geen nul waarden



In [ ]:
df = 

## Splitten van de data CHRONOLOGISCH
### De data splitten op chronologsiche volgorde. eerst train dan validatie en dan test, als je met shuffle =true werkt dan gaat het model al naar de toekomst kijken en dat mag niet
### Schalen van de data zodat de data tussen -3 en 3 ligt dit is beter voor het model
### De scaler leert het gemiddelde en afwijking alleen van de TRAIN data

In [ ]:
TARGET_IDX = FEATURE_COLS.INDEX(TARGET_COL)
N_FEATURES= len(FEATURE_COLS)

n=  len(df)
i_train, i_val = int(n * TRAIN_FRAC), int(n * (TRAIN_FRAC + VAL_FRAC))
values= df[FEATURE_COLS].values
# train test split op chronologische volgored
train_raw, val_raw, test_raw = values[:i_train], values[i_train:i_val], values[i_val:]

# scalen van de data (de scler word gefit op de TRAIN data!!!!!)
scaler = StandardScaler().fit(train_raw)
train = scaler.transform(train_raw).astype(np.float32)
val   = scaler.transform(val_raw).astype(np.float32)
test  = scaler.transform(test_raw).astype(np.float32)

NameError: name 'FEATURE_COLS' is not defined

# dataset (sliding window)

In [ ]:
class WindowDataset(Dataset):

    def __init__(self, data, input_len, horizon, target_idx, stride):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.input_len = input_len
        self.horizon = horizon
        self.target_idx = target_idx
        
        # Hoeveel vensters passen er in de data? Elk venster heeft input_len + horizon stappen nodig
        n_vensters = len(data) - input_len - horizon + 1
        # Startpositie van elk venster: 0, stride, 2*stride, ...
        self.starts = range(0, max(n_vensters, 0), stride)

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, i):
        s = self.starts[i]
        x = self.data[s : s + self.input_len]                                   # geschiedenis, alle kolommen
        y = self.data[s + self.input_len : s + self.input_len + self.horizon,   # toekomst,
                      self.target_idx]                                          # alleen de target
        return x, y
# de dataloader levert in batches 
train_dl = DataLoader(WindowDataset(train, imput_len, horizon, TARGET_IDX, stride), batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(WindowDataset(val,   imput_len, horizon, TARGET_IDX, stride), batch_size=BATCH_SIZE)
test_dl  = DataLoader(WindowDataset(test,  imput_len, horizon, TARGET_IDX, stride), batch_size=BATCH_SIZE)


NameError: name 'train' is not defined

# LSTM model


In [ ]:
class LSTM(nn.Module):
    def __init__ (self, n_features= N_FEATURES, hidden_size= HIDDEN_SIZE, num_layers = NUM_LAYERS, dropout= DROPOUT):
        super.__init__()
        self.rnn= nn.LSTM(input_size=n_features,
                        hidden_size=hidden_size,
                        num_layers=num_layers,
                        batch_first=True,
                        dropout=dropout if num_layers > 1 else 0.0)
        
        self.head = nn.Linear(hidden_size, horizon)
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1, :])

model = LSTM(N_FEATURES, HIDDEN_SIZE, NUM_LAYERS, DROPOUT, horizon)

SyntaxError: expected ':' (3936311493.py, line 2)

In [ ]:
def evaluate_loss(model, loader, loss_fn):
    # Gemiddelde loss berekenen
    model.eval()                        
    total = 0.0
    with torch.no_grad():              
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            total += loss_fn(model(x), y).item() * len(x)
    return total / len(loader.dataset)

def train_model(model, train_loader, val_loader):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)   
    loss_fn = nn.L1Loss()                               # MAE

    best_val = float("inf")                             
    best_state = None                                   
    history = {"train": [], "val": []}                  # voor de plot

    for epoch in range(EPOCHS):
        model.train()                                   
        train_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()                             
            loss = loss_fn(model(x), y)                 
            loss.backward()                             
            opt.step()                                  
            train_loss += loss.item() * len(x)
        train_loss /= len(train_loader.dataset)         # loss

        val_loss = evaluate_loss(model, val_loader, loss_fn)   

        history["train"].append(train_loss)
        history["val"].append(val_loss)
        print(f"epoch {epoch + 1:02d} | train {train_loss:.4f} | val {val_loss:.4f}")

        # Is dit het beste model tot nu toe? Dan bewaren we de gewichten
        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)                   # terug naar het beste model
    return model, history


In [ ]:
start = time.time()
model, history = train_model(model, train_dl, val_dl)
print(f"trainingstijd: {time.time() - start:.1f} seconden")

In [9]:
def test_per_dag(model, loader):
    model.eval()
    voorspellingen, werkelijk = [], []
    with torch.no_grad():
        for x, y in loader:
            voorspellingen.append(model(x.to(DEVICE)).cpu())
            werkelijk.append(y)
    voorspellingen = torch.cat(voorspellingen).numpy()      # vorm (aantal vensters, horizon)
    werkelijk = torch.cat(werkelijk).numpy()

    # Terugrekenen naar ongeschaald
    gemiddelde = scaler.mean_[TARGET_IDX]
    spreiding = scaler.scale_[TARGET_IDX]
    voorspellingen = voorspellingen * spreiding + gemiddelde
    werkelijk = werkelijk * spreiding + gemiddelde

    # Fout per dag dag 1 = stappen 0-287, dag 2 = 288-575, dag 3 = 576-863
    rijen = []
    for dag in range(aantal_dagen_forecast):
        stuk = slice(dag * stap_per_dag, (dag + 1) * stap_per_dag)
        fout = voorspellingen[:, stuk] - werkelijk[:, stuk]
        rijen.append({
            "dag": dag + 1,
            "MAE": np.abs(fout).mean(),
            "RMSE": np.sqrt((fout ** 2).mean()),
        })
    return pd.DataFrame(rijen).set_index("dag"), voorspellingen, werkelijk

scores, voorspellingen, werkelijk = test_per_dag(model, test_dl)
scores

NameError: name 'model' is not defined

In [ ]:
# Leercurve: gaat de fout omlaag? En blijft de validatiefout meedalen?
plt.figure(figsize=(7, 3.5))
plt.plot(history["train"], label="train")
plt.plot(history["val"], label="validatie")
plt.xlabel("epoch")
plt.ylabel("fout (MAE, geschaald)")
plt.legend()
plt.show()

In [ ]:
# Voorbeeld: de voorspelling tegenover de werkelijke waarden voor het eerste testvenster
plt.figure(figsize=(11, 3.5))
plt.plot(werkelijk[0], label="werkelijk", color="black")
plt.plot(voorspellingen[0], label="voorspelling")
for dag in range(1, aantal_dagen_forecast):             # stippellijnen tussen de dagen
    plt.axvline(dag * stap_per_dag, linestyle="--", color="grey", linewidth=0.8)
plt.xlabel("stappen vooruit (1 stap = 5 minuten)")
plt.legend()
plt.show()